# Лабораторна Робота №7
## Тема: Структури даних дерево, купа, гештаблиця
### Мета: засвоїти основні функції та алгоритми роботи з деревами та купою засобами Python.
#### Виконав: Гнатюк Євгеній

# Порядок виконання роботи
## Завдання 1:
- Створити бінарне дерево згідно з варіантом, виданим викладачем.
- Написати процедуру видалення заданої гілки дерева.
-  Оцінити асимптотичну складність (в середньому і в найгіршому
випадку) процедур search, insert і delete роботи з деревом.


In [1]:
def create_node(key):
    return {'key': key, 'left': None, 'right': None}

def insert(root, key):
    if root is None:
        return create_node(key)
    if key < root['key']:
        root['left'] = insert(root['left'], key)
    elif key > root['key']:
        root['right'] = insert(root['right'], key)
    
    return root

def search(root, key):
    if root is None:
        return False
    if root['key'] == key:
        return True

    if key < root['key']:
        return search(root['left'], key)
    else:
        return search(root['right'], key)

def inorder_traversal(root, result=None):
    if result is None:
        result = []
        
    if root is not None:
        inorder_traversal(root['left'], result)
        result.append(root['key'])
        inorder_traversal(root['right'], result)
        
    return result

if __name__ == "__main__":
    tree_root = None
    elements = [50, 30, 20, 40, 70, 60, 80]
    for el in elements:
        tree_root = insert(tree_root, el)
        
    print("Дерево після вставлення (Inorder обхід):")
    print(inorder_traversal(tree_root))
    search_key = 60
    if search(tree_root, search_key):
        print(f"Елемент {search_key} знайдено в дереві.")
    else:
        print(f"Елемент {search_key} відсутній.")

Дерево після вставлення (Inorder обхід):
[20, 30, 40, 50, 60, 70, 80]
Елемент 60 знайдено в дереві.


Вставлення та Пошук: 
- Середній випадок: $O(\log n)$ (дерево збалансоване).
- Найгірший випадок: $O(n)$ (елементи вставляються по порядку, дерево вироджується у звичайний список).

Обхід дерева (Inorder):
- $O(n)$ (необхідно відвідати кожен з $n$ вузлів).

Просторова складність:
- $O(n)$ (зберігання самих елементів) + від $O(\log n)$ до $O(n)$ додаткової пам'яті для стеку рекурсії.

 # Завдання 2:
- Написати процедуру генерації купи з будь-якого рандомного масива.
- Додати до нього елемент, який дорівнює вашому порядковому номеру у
списку групи.
- Вилучити максимальний елемент з купи
- Оцінити асимптотичну складність (у середньому і в найгіршому
випадку) процедур search, insert і delete роботи з купою.

In [2]:
import random

def heapify(arr, n, i):
    largest = i
    l, r = 2 * i + 1, 2 * i + 2
    if l < n and arr[l] > arr[largest]: largest = l
    if r < n and arr[r] > arr[largest]: largest = r
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        heapify(arr, n, largest)

def build_max_heap(arr):
    for i in range(len(arr) // 2 - 1, -1, -1):
        heapify(arr, len(arr), i)

def insert_max_heap(arr, item):
    arr.append(item)
    i = len(arr) - 1
    while i > 0 and arr[i] > arr[(i - 1) // 2]:
        arr[i], arr[(i - 1) // 2] = arr[(i - 1) // 2], arr[i]
        i = (i - 1) // 2

def extract_max(arr):
    if not arr: return None
    max_val = arr[0]
    arr[0] = arr[-1]
    arr.pop()
    if arr: heapify(arr, len(arr), 0)
    return max_val

arr = random.sample(range(1, 100), 10)
build_max_heap(arr)
insert_max_heap(arr, 4)
max_el = extract_max(arr)

print(f"Вилучений максимум: {max_el}")
print(f"Купа після операцій: {arr}")

Вилучений максимум: 98
Купа після операцій: [82, 79, 80, 75, 64, 4, 71, 31, 19, 57]


Пошук (Search): $O(n)$. Купа не є повністю відсортованою, тому для пошуку довільного елемента доведеться перевіряти всі вузли.

Вставлення (Insert): $O(\log n)$ у найгіршому випадку (елемент піднімається до кореня), $O(1)$ у середньому.

Вилучення максимуму (Delete): $O(\log n)$ (у середньому і в найгіршому). Корінь замінюється останнім елементом, який потім "просіюється" вниз на глибину дерева ($\log n$).

# Завдання 3:
1. Вивчіть самостійно різні методи розв’язання колізій.
2. Реалізуйте геш-таблицю з ланцюжковим гешуванням.
3. Проведіть тестування геш-таблиці з різними типами даних (цілі числа,
рядки, списки, словники, об’єкти):
- перевірити працездатність геш-таблиці з різними типами даних.
- виміряти час виконання основних операцій (пошук, вставка, видалення)
для різних типів даних.
- порівняти результати для різних типів даних.

In [4]:
import time

class HashTable:
    def __init__(self, size=100):
        self.size = size
        # Створюємо масив порожніх списків (ланцюжків)
        self.table = [[] for _ in range(size)]

    def _to_hashable(self, key):
        """Перетворює змінні типи (list, dict) у незмінні для гешування."""
        if isinstance(key, list):
            return tuple(key)
        if isinstance(key, dict):
            return frozenset(key.items())
        return key

    def _hash(self, key):
        """Обчислює індекс за допомогою вбудованої функції hash()."""
        hashable_key = self._to_hashable(key)
        return hash(hashable_key) % self.size

    def insert(self, key, value):
        index = self._hash(key)
        hashable_key = self._to_hashable(key)
    
        for item in self.table[index]:
            if item[0] == hashable_key:
                item[1] = value
                return
        self.table[index].append([hashable_key, value])

    def search(self, key):
        index = self._hash(key)
        hashable_key = self._to_hashable(key)
        
        for item in self.table[index]:
            if item[0] == hashable_key:
                return item[1]
        return None

    def delete(self, key):
        index = self._hash(key)
        hashable_key = self._to_hashable(key)
        
        for i, item in enumerate(self.table[index]):
            if item[0] == hashable_key:
                del self.table[index][i]
                return True
        return False

class CustomObject:
    def __init__(self, name):
        self.name = name

def measure_time(ht, key, value):
    t0 = time.perf_counter()
    ht.insert(key, value)
    t_insert = time.perf_counter() - t0
    t0 = time.perf_counter()
    ht.search(key)
    t_search = time.perf_counter() - t0
    t0 = time.perf_counter()
    ht.delete(key)
    t_delete = time.perf_counter() - t0
    
    return t_insert, t_search, t_delete

if __name__ == "__main__":
    ht = HashTable()

    test_data = {
        "Integer (Ціле число)": (123456, "Value for Int"),
        "String (Рядок)": ("my_string_key", "Value for Str"),
        "List (Список)": ([1, 2, 3], "Value for List"),
        "Dict (Словник)": ({"a": 1, "b": 2}, "Value for Dict"),
        "Object (Об'єкт класу)": (CustomObject("test"), "Value for Obj")
    }
    
    print(f"{'Тип даних':<22} | {'Вставка (с)':<15} | {'Пошук (с)':<15} | {'Видалення (с)':<15}")
    print("-" * 75)
    
    for type_name, (key, value) in test_data.items():
        t_ins, t_search, t_del = measure_time(ht, key, value)
        print(f"{type_name:<22} | {t_ins:<15.7f} | {t_search:<15.7f} | {t_del:<15.7f}")

Тип даних              | Вставка (с)     | Пошук (с)       | Видалення (с)  
---------------------------------------------------------------------------
Integer (Ціле число)   | 0.0000082       | 0.0000026       | 0.0000053      
String (Рядок)         | 0.0000040       | 0.0000020       | 0.0000024      
List (Список)          | 0.0000029       | 0.0000025       | 0.0000023      
Dict (Словник)         | 0.0000036       | 0.0000025       | 0.0000029      
Object (Об'єкт класу)  | 0.0000025       | 0.0000014       | 0.0000016      



# Контрольні запитання:
1. Бінарне дерево просто обмежує кількість нащадків до двох. У БДП додатково діє правило: всі значення в лівому піддереві менші за батька, а в правому — більші.
2. 
Купа — це повне бінарне дерево, яке зберігається у вигляді звичайного одновимірного масиву. Крім того, в купі діє властивість: батьківський елемент завжди більший (або менший) за своїх нащадків.

3. Бінарні дерева пошуку (швидкий пошук, додавання), збалансовані дерева (АВЛ, червоно-чорні — запобігають перетворенню дерева на лінійний список, гарантуючи $O(\log n)$), B-дерева (використовуються в базах даних).

4. Організація ієрархічних даних (файлові системи комп'ютера), побудова швидких індексів у базах даних, оптимальне кодування даних (дерево Гафмена).

5. Купа зберігається як масив.  Додавання: елемент ставиться в самий низ купи (в кінець масиву) і міняється місцями з батьківським, поки не стане на своє місце (спливає). Вилучення (максимуму): перший (кореневий) елемент видаляється, на його місце ставиться самий нижній (останній в масиві), після чого він "просіюється" вниз до правильної позиції.

6. Реалізація ефективних черг з пріоритетами (де потрібно швидко діставати елемент з найвищим пріоритетом), алгоритм пірамідального сортування (HeapSort), алгоритми на графах (наприклад, Дейкстри).

8. Геш-функція перетворює вхідний ключ на ціле число. Це число використовується як індекс у масиві , що дозволяє миттєво (за $O(1)$) знаходити, додавати або видаляти значення.
9. Метод ланцюжків: у кожній комірці масиву зберігається зв'язний список елементів. Перевага: легко реалізувати, Недолік: потребує додаткової пам'яті.Відкрита адресація: якщо комірка зайнята, елемент записується у наступну вільну. Перевага: не потрібна додаткова пам'ять, Недолік: таблиця може "засмічуватися" і працювати повільніше при сильному заповненні.